# CVAI · Train E5 truy xuất và MLP gán nhãn
Chọn runtime GPU T4, rồi chạy các ô theo thứ tự. Chọn gói `cvai_retrieval_r1.zip` khi Colab hỏi. Kết quả của mỗi lần chạy được lưu vào thư mục riêng trên Google Drive. Dữ liệu CV và JD trong gói là tiếng Anh, tổng hợp.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, subprocess, sys, zipfile
drive.mount('/content/drive')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Chọn một file ZIP của dự án")
import tempfile
work = Path(tempfile.mkdtemp(dir='/content'))
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    archive.extractall(work)
os.chdir(work)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## Chạy hai cấu hình
Cùng dữ liệu R1 và MLP 64 nút. Cấu hình thứ hai cập nhật hai lớp Transformer cuối của E5 bằng các bộ ba yêu cầu, đoạn đúng và đoạn gây nhiễu. Mỗi cấu hình tự truy xuất top 3 trước khi train MLP. Chọn theo macro-F1 validation, ưu tiên bản đóng băng khi bằng điểm.

In [ ]:
from datetime import datetime, timezone
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU trong Runtime / Change runtime type")
run = Path('/content/drive/MyDrive/FutureCV-ML') / datetime.now(timezone.utc).strftime('retrieval-r1-%Y%m%dT%H%M%S%fZ')
print(run, flush=True)
subprocess.run([sys.executable, '-u', 'train_retrieval_mlp.py', '--data-dir', 'data',
                '--pdf-dir', 'pdf', '--output-dir', str(run), '--device', 'cuda'], check=True)

## Đọc kết quả
`comparison.json` ghi hai kết quả validation. `pdf_evaluation.json` ghi từng nhãn dự đoán và ba đoạn PDF tương ứng của model đã chọn. Thư mục `frozen` hoặc `tuned` chứa trọng số encoder, tokenizer, MLP và manifest.

In [ ]:
import json
comparison = json.loads((run/'comparison.json').read_text())
pdf = json.loads((run/'pdf_evaluation.json').read_text())
for name, m in [*comparison['validation'].items(), ('PDF diagnostic', pdf)]:
    print(name, {k: round(m[k], 4) for k in ('macro_f1','span_recall_at_3','score_mae')})
print('Model đã chọn:', run/comparison['selected'])